<a href="https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CevdetSatarr/FlyRank-intern/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Task type: Ranking / scoring**, not classification and not clustering.

Lane 2 (Refresh / Content Opportunity Scoring) asks *which pages should be reviewed first* — that's an ordering question, not a yes/no label. A binary classifier would tell me a page is 'declining' or not, but 54% of pages already carry `trend_direction == 'down'` (per the ML-02 baseline), so a classifier alone gives editors a pile of over 16,000 pages with no way to choose where to start. What the review team actually needs is a **ranked queue**: order all declining pages by how much opportunity is actually recoverable, so the top 20–50 are worth a human's next hour.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Proxy, not an observed outcome.** The starter dataset already ships a directional proxy: `trend_direction` and `trend_pct`, built by comparing `clicks_last_30d`/`impressions_last_30d` against `clicks_prev_30d`/`impressions_prev_30d` — a defined rule (a 30-day-over-30-day comparison), not a ground-truth label of 'this page will recover if refreshed.'

I will build an **opportunity score** on top of that proxy: rank pages where `trend_direction == 'down'` **and** `search_volume` is meaningful **and** `avg_position` is still close enough to page one to be recoverable (e.g. `position_tier` in `striking` or `page_1`) higher than declining pages with near-zero demand or a position too deep to realistically fix with a refresh. The label the model would ultimately be scored against is still a proxy: whether a human reviewer, looking at the top of the queue, agrees the page is actually worth prioritizing — not a guarantee that refreshing it will recover traffic.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Precision@20** — of the top 20 pages my ranking puts at the top of the queue, what fraction would a human reviewer actually agree belong there? This is the metric the lane guide names directly, and it matches how the output will really be used: editors don't review a ranked list end-to-end, they work from the top down until their hour runs out. A high Precision@20 means their limited time isn't wasted; a model that's accurate on average but wrong at the top of the list is useless here even if its overall accuracy looks fine.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [8]:
import pandas as pd

# Load the anonymized starter dataset for Lane 2 (matches the file used in ML-02)
df = pd.read_csv('/content/content_refresh_anonymized.csv')

print('shape:', df.shape)
print()
print(df[['content_id', 'client_id', 'search_volume', 'avg_position', 'position_tier',
          'ctr', 'trend_direction', 'trend_pct', 'freshness_tier']].head())


shape: (30000, 44)

             content_id          client_id  search_volume  avg_position  \
0  content_304f48230142  client_f369cb89fc           10.0          10.6   
1  content_a1fb4e703a9e  client_4e07408562           90.0          20.3   
2  content_9aa793d4d895  client_7f2253d7e2            0.0          36.5   
3  content_331d6c4de07b  client_19581e27de           10.0           6.2   
4  content_d99b7a2d90ca  client_3fdba35f04            0.0          44.0   

  position_tier   ctr trend_direction  trend_pct freshness_tier  
0      striking  0.76            down      -41.4           0-30  
1      page_3_5  0.05            down      -57.7           0-30  
2      page_3_5  0.09            down      -60.9           0-30  
3        page_1  0.49          stable      -13.8           0-30  
4      page_3_5  0.13            down      -34.7           0-30  


**One row = one pseudonymized content item** (`content_id`), with its 90-day and 30-day windowed search/engagement history and a handful of pre-computed tiers (`position_tier`, `freshness_tier`, `age_tier`). 30,000 rows, one per content item — matches the ML-02 baseline exactly.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [9]:
# A single if-statement rule (trend_direction == 'down') can't tell these two pages apart —
# both 'decline', but one is a real opportunity and one almost certainly isn't.
declining = df[df['trend_direction'] == 'down']

high_opportunity = declining[(declining['search_volume'] > declining['search_volume'].median()) &
                              (declining['position_tier'].isin(['striking', 'page_1']))]
low_opportunity  = declining[(declining['search_volume'] <= declining['search_volume'].median()) &
                              (declining['position_tier'].isin(['deep', 'page_3_5']))]

print(f"Declining pages total: {len(declining):,}")
print(f"  - high-opportunity slice (real demand, close to page 1): {len(high_opportunity):,}")
print(f"  - low-opportunity slice (low demand, buried deep): {len(low_opportunity):,}")
print()
print('A single rule (trend_direction == down) treats all of these as equally worth reviewing.')
print('These two slices show that is false — same rule outcome, very different real opportunity.')


Declining pages total: 16,262
  - high-opportunity slice (real demand, close to page 1): 3,232
  - low-opportunity slice (low demand, buried deep): 3,140

A single rule (trend_direction == down) treats all of these as equally worth reviewing.
These two slices show that is false — same rule outcome, very different real opportunity.


**Why this needs ranking, not a rule:** `trend_direction == 'down'` fires on 54% of the inventory and, on its own, can't separate a page with real recoverable demand from one that's declining because nobody was ever searching for it. Telling those apart means weighing several continuous signals against each other at once — `search_volume`, `avg_position`, `ctr`, `freshness_tier`, `content_age_days` — where the right threshold for one depends on the value of the others (a striking-tier page with low volume may still outrank a page-1 page with high volume, depending on the CTR gap). That kind of multi-signal, interacting tradeoff is exactly what an if-else chain can't hold, and what a learned or transparent weighted score can.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.